# Le prix de l'immobilier dépend-il de l'accès aux commerces alimentaires ?

*Projet de Python pour la data science (ENSAE, 2026) — Antoine Charki, Samuel Barre*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from src import config

libelles_dens7 = {
    1: "1 Grands centres urbains", 2: "2 Centres urbains intermédiaires", 3: "3 Ceintures urbaines",
    4: "4 Petites villes", 5: "5 Bourgs ruraux", 6: "6 Rural à habitat dispersé",
    7: "7 Rural à habitat très dispersé",
}

def lire(nom):
    """Lit un tableau de résultats produit par les notebooks de traitement."""
    return pd.read_csv(config.DOSSIER_TABLES / nom)

def montrer(nom, largeur=600):
    """Affiche une figure produite par les notebooks de traitement."""
    chemin = config.DOSSIER_FIGURES / nom
    if chemin.exists():
        display(Image(chemin, width=largeur))
    else:
        print(f"Figure manquante : {nom} (lancer le notebook qui la produit)")

## 1. La question

Le prix d'un logement dépend de sa surface, de son état, de sa localisation… et peut-être aussi
de la facilité à faire ses courses. On se demande si le prix au m² des logements vendus en 2025
est associé à l'accès aux commerces alimentaires, en distinguant quatre types de commerces
(hypermarchés, supermarchés, commerces de proximité et commerces spécialisés), **à type de territoire égal**.

Cette dernière précision est essentielle : un appartement parisien est à la fois cher et entouré
de commerces, une maison rurale est à la fois bon marché et loin de tout. Comparer directement
les deux mesurerait surtout l'opposition ville / campagne. On compare donc des logements situés dans
des communes de même densité.

## 2. Les données

| Source | Contenu | Utilisation |
|---|---|---|
| Base permanente des équipements (BPE) 2025, INSEE | Tous les équipements de France, localisés | Commerces alimentaires (domaines B1 et B2) |
| Demandes de valeurs foncières (DVF) géolocalisées 2025, Etalab / DGFiP | Toutes les ventes immobilières, avec prix, surface et coordonnées | Prix au m² des logements vendus |
| Grille communale de densité 2025, INSEE | Niveau de densité de chaque commune, en 7 niveaux | Comparer à type de territoire égal |

Les trois fichiers sont téléchargés directement depuis leur source publique par les notebooks de
traitement. Les deux bases sont croisées géographiquement avec `geopandas` : pour chaque vente, on
calcule la distance au commerce le plus proche de chaque type.

Le champ d'étude est la France métropolitaine **hors Alsace-Moselle**, que la DVF ne couvre pas (ces
départements ont leur propre système de publicité foncière, le Livre foncier).

## 3. Le nettoyage et les difficultés rencontrées

### 3.1 Les commerces (notebook `01_nettoyage_bpe`)

| Étape | Commerces retirés |
|---|---|
| Commerces alimentaires uniquement (domaines B1 et B2, sans les grandes surfaces de bricolage B103) | — |
| Commerces sans coordonnées (dont 1 784 à Mayotte) | 4 756 |
| DROM et Alsace-Moselle | 10 968 |
| **Base finale** | **135 587 commerces, 11 types** |

**Délimiter les commerces alimentaires.** La BPE recense tous les équipements (écoles, médecins, commerces…).
On n'a gardé que les grandes surfaces et les commerces alimentaires (domaines B1 et B2), en retirant
les grandes surfaces de bricolage, classées dans le même domaine que les hypermarchés.

**Les commerces sans coordonnées et le cas du 75116.** 4 756 commerces n'ont pas de position
géographique. Un peu plus d'un tiers sont à Mayotte, qui sort de toute façon du champ. Mais 119 sont à
Paris, et ils n'ont ni coordonnées ni code postal.

Pour savoir si ces manques étaient répartis au hasard, on a compté les commerces de Paris par code
postal. Le résultat contenait une anomalie flagrante : le code postal 75116, qui couvre le nord du
16e arrondissement, ne comptait qu'**un seul commerce alimentaire**. C'est invraisemblable pour un
quartier dense et résidentiel. Les commerces sans coordonnées sont donc probablement concentrés dans
quelques quartiers, et pas répartis au hasard.

Nous avons interrogé notre enseignant, qui nous a proposé quatre approches :
1. **Comparer le nord et le sud du 16e** (75116 et 75016) à partir d'autres équipements et des loyers.
   Si leurs distributions sont proches, on peut considérer que les deux zones ont des caractéristiques
   similaires et s'appuyer uniquement sur le 75016.
2. **Compléter les données à la main**, en recherchant les commerces manquants sur Google Maps ou une
   autre source, puis en les recoupant avec la BPE.
3. **Renoncer au 75116**, et indiquer clairement que l'analyse ne peut pas y être menée, ou seulement
   sous certaines conditions. C'est d'autant plus défendable que ce quartier, très aisé, est
   probablement atypique.
4. **Renoncer à l'analyse par type de commerce**, ce qui n'a de sens que si les types de commerces ne
   révèlent rien de particulier.

Nous avons retenu la troisième approche. Plutôt que d'inventer des positions, nous retirons ces
119 commerces (moins de 0,1 % de la base), et nous signalons que les résultats concernant le nord du
16e arrondissement doivent être lus avec précaution : l'accès aux commerces y est sous-estimé.
La quatrième approche était exclue, puisque les résultats diffèrent nettement selon le type de
commerce (partie 6). La deuxième aurait demandé un travail manuel important et difficile à
reproduire. La première demandait des données supplémentaires (loyers, autres équipements) qu'il
aurait fallu collecter et nettoyer en plus.


**L'outre-mer et l'Alsace-Moselle.** On a d'abord travaillé sur toute la France, avant de restreindre
le champ à la métropole hors Alsace-Moselle, pour trois raisons :
- la DVF ne couvre ni l'Alsace-Moselle (Livre foncier) ni Mayotte : sans prix, ces commerces ne servent à rien ;
- les distances sont calculées en Lambert 93, une projection valable uniquement pour la métropole ;
- les marchés immobiliers et commerciaux des DROM, insulaires, ne se comparent pas à ceux de la métropole.

**Les « hypermarchés » de Paris sont des grands magasins.** La BPE recense 44 hypermarchés dans Paris,
mais leur code d'activité montre qu'il s'agit presque tous de grands magasins (Samaritaine, BHV,
Galeries Lafayette, Printemps, Bon Marché). C'est pourquoi les résultats sont aussi présentés hors Paris.

**Un magasin peut compter plusieurs fois.** La BPE compte des établissements (SIRET), pas des magasins :
le BHV apparaît ainsi sur 8 lignes. Les doublons stricts (même type, même nom, même position) ne
représentent que 0,07 % de la base et ne modifient pas les distances : on les conserve.

### 3.2 Les ventes immobilières (notebook `02_nettoyage_dvf`)

| Étape | Ventes retirées | Raison |
|---|---|---|
| Ventes comprenant plusieurs logements ou un local professionnel | 60 855 | Le prix ne peut pas être attribué à un seul logement |
| Coordonnées hors de la métropole (surtout les DROM) | 9 242 | Hors du champ |
| Autres natures que « Vente » (adjudications, échanges…) | 10 902 | Prix pas comparables |
| Surface inférieure à 9 m² ou prix manquant | 358 | Pas un logement habitable |
| Prix inférieur à 100 €/m² | 2 651 | Ventes hors marché |
| Prix supérieur à 10 fois la médiane de la commune la plus chère (Saint-Tropez) | 6 | Erreurs de saisie |
| Prix à moins d'un dixième ou plus de 10 fois la médiane de la commune | 1 954 | Valeurs aberrantes pour la commune |
| **Base finale** | **810 517 ventes** | |

**Avant les filtres : trouver la bonne version de la DVF.** On a d'abord téléchargé la DVF brute publiée
par la DGFiP (fichier texte), qui ne contient pas de coordonnées : impossible d'y mesurer une distance.
On est passés à la version géolocalisée d'Etalab, qui donne la longitude et la latitude de chaque vente.

**Comprendre la structure de la base.** Une même vente occupe plusieurs lignes, une par parcelle et par
local (maison, dépendance, terrain…). Les nombreuses valeurs manquantes sont donc normales : une parcelle
sans bâtiment n'a pas de surface bâtie. On n'a gardé que les maisons et appartements, en ne comptant
qu'une fois un logement répété sur plusieurs parcelles.

**Étape 1 : les ventes de plusieurs logements (60 855).** Quand une vente comprend plusieurs logements
(un immeuble, un lot de studios) ou un logement avec un local commercial, la DVF reporte le prix **total**
sur chaque ligne. Diviser ce prix par la surface d'un seul logement donne un prix au m² très surévalué.
Le problème n'a été repéré qu'en examinant les médianes par commune : Crégy-lès-Meaux affichait un prix
médian absurde. C'est de loin le filtre qui retire le plus de ventes, et le plus important pour la
qualité des prix.

**Étape 2 : les coordonnées hors de la métropole (9 242).** On ne garde que les ventes situées dans un
rectangle englobant la métropole et la Corse. Il s'agit presque uniquement de ventes dans les DROM,
retirées pour les mêmes raisons que les commerces (partie 3.1).

**Étape 3 : les autres natures de mutation (10 902).** La DVF contient aussi des ventes en l'état futur
d'achèvement (logements neufs sur plan), des adjudications (ventes aux enchères, souvent judiciaires), des
échanges et des expropriations. Leurs prix obéissent à d'autres logiques que le marché de l'ancien : on ne
garde que les ventes classiques.

**Étape 4 : les surfaces de moins de 9 m² et les prix manquants (358).** En dessous de 9 m², un logement
n'est pas décent au sens de la loi : ce sont des chambres de service, des erreurs de saisie ou des
surfaces mal renseignées. Diviser un prix par une surface minuscule donne de plus des prix au m² extrêmes.

**Étape 5 : le plancher de 100 €/m² (2 651, soit 0,33 %).** Ces ventes ne sont pas des erreurs, mais
elles ne reflètent pas un prix de marché :
- des ventes à l'euro symbolique, souvent entre collectivités, bailleurs sociaux ou entreprises ;
- des ventes entre proches à un prix « d'ami » ;
- des rachats de parts entre héritiers après une succession, où le prix ne porte que sur une partie du bien ;
- des viagers, pour lesquels la DVF n'enregistre que le capital versé à la signature (le « bouquet »), et pas les rentes ;
- des maisons en ruine ou inhabitables, vendues pour le terrain.

La moitié de ces ventes se fait à moins de 2 000 € au total, et un quart à moins de 500 €. Les garder
tirerait vers le bas les prix des communes rurales, où ces ventes sont proportionnellement plus fréquentes.

**Étape 6 : le plafond (6).** Notre première idée était un seuil fixe (retirer les ventes au-dessus de
50 millions d'euros). Mais un seuil national n'a pas de sens : un prix normal dans le 6e arrondissement de
Paris serait aberrant dans la Creuse. On a donc fixé le plafond à 10 fois la médiane de la commune la plus
chère de France ayant au moins 100 ventes, Saint-Tropez (15 170 €/m²), soit environ 150 000 €/m². Ce plafond est volontairement large pour éviter de retirer des biens de communes ou quartiers exceptionnels qui ne sont pas pour autant des erreurs de saisie. On avait
d'abord pensé au 6e arrondissement de Paris. Les 6 ventes retirées sont manifestement des erreurs de saisie
(prix ou surface décalés d'un ou plusieurs zéros).

**Étape 7 : le filtre relatif à la commune (1 954).** Un prix peut être plausible à l'échelle nationale
et aberrant à l'échelle locale. On retire donc les ventes à plus de 10 fois ou moins d'un dixième de la
médiane de leur commune. On l'a fait au niveau de la commune plutôt que du code postal, parce que c'est à
cette échelle que la DVF et la grille de densité sont définies.

**L'ordre des filtres compte.** Les ventes à prix symbolique tiraient vers le bas la médiane de
certaines communes (Vitreux), ce qui faussait le filtre relatif : de vraies ventes y paraissaient
anormalement chères. Le plancher est donc appliqué *avant* de calculer les médianes.

**Une fausse piste.** On a essayé de retirer les ventes à prix identiques au sein d'une commune, en
pensant à des ventes groupées mal découpées. Les cas trouvés étaient de vraies ventes (des studios
identiques à Lourdes), et le filtre a été abandonné.

**Le niveau de densité.** On a d'abord construit nous-mêmes un indicateur de densité, avant de le
remplacer par la grille communale de densité officielle de l'INSEE (7 niveaux). Elle est reconnue et
couvre toutes les communes. Seule adaptation : les arrondissements de Paris, Lyon et Marseille,
absents de la grille, reçoivent le niveau de leur commune.

## 4. Où sont les commerces ?

In [ ]:
montrer("repartition_commerces.png", largeur=450)

Les 135 587 commerces alimentaires se répartissent en quatre catégories très inégales. Les commerces
spécialisés (boulangeries, boucheries, poissonneries…) sont de loin les plus nombreux, devant les
commerces de proximité (supérettes et épiceries), les supermarchés et les hypermarchés.

In [ ]:
montrer("carte_commerces_france.png")

Les commerces suivent la répartition de la population : ils se concentrent dans les grandes
agglomérations (Paris, Lyon, Marseille, Lille, Toulouse…), le long des côtes et des vallées. Ils sont
beaucoup plus rares dans les zones peu peuplées (Massif central, Landes, intérieur de la Corse).
Les hypermarchés sont présents autour de presque toutes les villes, généralement en périphérie.
Paris forme pourtant une tache d'hypermarchés très dense : ce sont en réalité des grands magasins (voir partie 3).

In [ ]:
montrer("carte_commerces_paris.png", largeur=500)
montrer("carte_commerces_saint_etienne.png", largeur=500)

Le zoom sur deux agglomérations montre deux situations très différentes. À Paris et en petite
couronne, les commerces forment un tissu continu, et la plupart des logements ont tous les types de
commerces à proximité ; les hypermarchés se trouvent en banlieue. Dans l'agglomération stéphanoise,
les commerces se concentrent dans le centre-ville et le long de la vallée, et les grandes surfaces sont
en périphérie des zones bâties. Les conditions d'accès dépendent donc fortement du type de territoire,
ce qui justifie de construire l'indice séparément pour chaque niveau de densité.

## 5. Mesurer l'accès aux commerces

Pour chaque vente et chaque type de commerce, on calcule la distance au commerce le plus proche.
Une distance brute ne se compare pas d'un territoire à l'autre : 2 km, c'est loin à Paris et
proche dans un village. L'indice d'accessibilité est donc construit en trois temps :

1. **Un rayon par type de commerce et par niveau de densité** : le 99e percentile de la distance
   au commerce le plus proche. Au-delà de ce rayon, on considère qu'un commerce n'est plus accessible.
2. **Une décroissance linéaire** : chaque commerce situé dans le rayon compte d'autant plus qu'il est
   proche (poids 1 à distance nulle, 0 au bord du rayon). L'indice additionne ces poids.
3. **Un rang au sein de chaque niveau de densité** : chaque vente est classée de 0 (le moins bon
   accès de son niveau) à 1 (le meilleur). C'est ce rang qui permet de comparer à territoire égal.

Le choix du 99e percentile a été testé : les rangs obtenus avec le 95e et le 90e percentile sont très
corrélés aux nôtres (notebook `03_croisement`).

Formellement, pour un logement $i$ situé dans une commune de niveau de densité $k(i)$ et un type de commerce $c$ :

**Rayon.** Le rayon du niveau $k$ est le 99e percentile, parmi les logements de ce niveau, de la distance au commerce de type $c$ le plus proche :
$$R_{c,k} = q_{0{,}99}\Big(\min_{j \in c} d_{ij} \;\Big|\; k(i) = k\Big)$$

**Règle composite.** Un commerce $j$ ne compte pour le logement $i$ que s'il est dans le rayon des deux territoires, celui du logement et celui du commerce. Cette règle permet d'éviter que les centres périurbains aient un accès très fort aux commerces du grand centre urbain. Cela faussait les données, et dans les faits il est très difficile d'entrer dans une agglomération de forte densité :
$$R_{ij} = \min\big(R_{c,k(i)},\, R_{c,k(j)}\big)$$

**Indice d'accessibilité.** Chaque commerce du rayon apporte un poids qui décroît linéairement avec la distance, de 1 au pied du logement à 0 au bord du rayon :
$$A_{ic} = \sum_{j \in c} \max\left(0,\; 1 - \frac{d_{ij}}{R_{ij}}\right)$$

**Rang.** L'indice est enfin transformé en rang au sein du niveau de densité du logement, de 0 (le moins bon accès) à 1 (le meilleur) :
$$\text{rang}_{ic} = \frac{1}{n_{k(i)}} \,\#\big\{\, i' : k(i') = k(i),\; A_{i'c} \le A_{ic} \,\big\}$$

où $d_{ij}$ est la distance à vol d'oiseau (en mètres, en Lambert 93) entre le logement $i$ et le commerce $j$, et $n_k$ le nombre de logements du niveau $k$.

Un commerce situé au pied du logement compte pour 1, un commerce à mi-rayon pour 0,5, et un commerce au-delà du rayon pour 0 : l'indice se lit comme un nombre de commerces « en équivalent pied d'immeuble ».

Dans le notebook `03_croisement`, ce calcul est fait par la fonction ci-dessous, appelée pour chaque type de commerce et chaque niveau de densité. Elle repose sur un arbre de recherche spatiale (`scipy.spatial.cKDTree`), qui trouve directement tous les couples logement-commerce à moins d'une distance donnée, sans calculer les 810 000 × 135 000 distances possibles. Les logements sont traités par blocs de 5 000 pour limiter la mémoire utilisée.

```python
def indice_lineaire(xy_ventes, rayon_vente, arbre, rayons_commerces, comptage=False):
    """Somme sur les commerces de max(0, 1 - d/R), avec R = min(rayon du logement, rayon du commerce)."""
    resultat = np.zeros(len(xy_ventes))
    for debut in range(0, len(xy_ventes), TAILLE_BLOC):
        bloc = xy_ventes[debut:debut + TAILLE_BLOC]
        # Couples (logement i, commerce j) à moins de R_{c,k(i)}, avec leur distance d_ij
        paires = cKDTree(bloc).sparse_distance_matrix(arbre, max_distance=rayon_vente, output_type="ndarray")
        # Règle composite : R_ij = min(R_{c,k(i)}, R_{c,k(j)})
        rayon_paire = np.minimum(rayon_vente, rayons_commerces[paires["j"]])
        if comptage:
            # Variante : nombre de commerces dans le rayon
            contributions = (paires["v"] < rayon_paire).astype(int)
        else:
            # Poids linéaire : max(0, 1 - d_ij / R_ij)
            contributions = np.clip(1 - paires["v"] / rayon_paire, 0, None)
        # Somme des poids pour chaque logement : A_ic
        resultat[debut:debut + len(bloc)] = np.bincount(paires["i"], weights=contributions, minlength=len(bloc))
    return resultat
```

L'option `comptage=True` donne une variante plus simple de l'indice, le nombre de commerces dans le rayon, $N_{ic} = \#\{\, j \in c : d_{ij} < R_{ij} \,\}$ (frontière nette). Elle est utilisée dans le modèle M5 de la partie 7.

In [ ]:
rayons = lire("rayons.csv")
rayons["DENS7"] = rayons["DENS7"].map(libelles_dens7)
(rayons.set_index("DENS7") / 1000).round(1).rename_axis("Niveau de densité").style.format("{:.1f} km")

Les rayons augmentent fortement avec la ruralité. Dans les grands centres urbains, presque tous les
logements ont un hypermarché à moins de 4,6 km et un commerce spécialisé à moins de 1 km. Dans le rural
à habitat très dispersé, ces rayons atteignent 38 km et 9 km.

Les ceintures urbaines ont un rayon plus grand que les petites villes pour tous les types de commerce, mais leurs
distances médianes sont plus faibles (partie 6). Le logement typique d'une ceinture urbaine
est donc mieux desservi : le grand rayon traduit la présence d'une minorité de logements très éloignés,
dans les communes périurbaines les plus étendues.

## 6. Prix, distance et densité

In [ ]:
montrer("repartition_ventes_densite.png", largeur=450)

Plus d'un tiers des ventes (38 %) a lieu dans les grands centres urbains, mais un autre tiers (33 %)
se fait dans des communes rurales (bourgs ruraux, rural dispersé et très dispersé) : le marché
immobilier ne se limite pas aux métropoles.

In [ ]:
montrer("prix_median_densite.png")

Les grands centres urbains (`DENS7 = 1`) présentent les prix médians au m² les plus élevés, ce n'est pas vraiment une surprise.

Plus étonnant, les petites villes (`DENS7 = 4`) arrivent en deuxième position, devant les centres urbains
intermédiaires et les ceintures urbaines. Ce classement peut en partie venir de la composition des
ventes (part de maisons et d'appartements, villes littorales ou touristiques parmi les petites villes),
que ce graphique ne distingue.


In [ ]:
distances = lire("distances_par_densite.csv")
distances["DENS7"] = distances["DENS7"].map(libelles_dens7)
distances.set_index("DENS7").rename_axis("Niveau de densité").round(0)

La distance médiane au commerce le plus proche augmente nettement avec la ruralité, pour les quatre
types de commerces. Dans les grands centres urbains, la moitié des logements vendus a un commerce
spécialisé à moins de 160 m et un hypermarché à moins de 1,2 km. Dans le rural à habitat très
dispersé, ces distances passent à 3,3 km et 15 km. Comparer directement les prix selon la distance
reviendrait donc surtout à comparer la ville et la campagne : c'est pourquoi l'analyse se fait à
niveau de densité égal.

In [ ]:
def graphique_tranches(nom_fichier, titre):
    """Prix médian par tranche d'accès, relatif à la tranche la moins bien desservie, par niveau de densité."""
    tranches = lire(nom_fichier)
    tranches["prix_relatif"] = 100 * tranches["prix_m2_median"] / tranches.groupby(
        ["categorie", "DENS7"])["prix_m2_median"].transform("first")

    fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
    for ax, (categorie, donnees) in zip(axes, tranches.groupby("categorie")):
        for dens7, d in donnees.groupby("DENS7"):
            ax.plot(d["tranche_acces"], d["prix_relatif"], marker="o", label=libelles_dens7[dens7])
        ax.set_title(categorie)
        ax.set_xlabel("Tranche d'accès (1 = le plus faible)")
        ax.set_xticks([1, 2, 3, 4, 5])
    axes[0].set_ylabel("Prix médian au m² (tranche 1 = 100)")
    axes[-1].legend(fontsize=7, loc="upper left", bbox_to_anchor=(1, 1))
    fig.suptitle(titre)
    plt.tight_layout()
    plt.show()

graphique_tranches("prix_par_tranche.csv", "France métropolitaine hors Alsace-Moselle")

Les tranches d'accès sont les quintiles du rang de l'indice d'accessibilité (partie 5) au sein de chaque niveau de densité : la tranche 1 regroupe les 20 % de logements les moins bien desservis de leur niveau, la tranche 5 les 20 % les mieux desservis.

Chaque courbe représente un niveau de densité. Le prix médian de la tranche la moins bien desservie
vaut 100, ce qui permet de comparer les niveaux entre eux. Une courbe qui monte signifie qu'à densité
égale, les logements les mieux desservis se vendent plus cher.

**Les commerces de proximité et spécialisés sont associés à des prix plus élevés.** Pour presque tous
les niveaux de densité, le prix augmente avec l'accès. La tranche la mieux desservie se vend 35 % à 85 %
plus cher que la moins bien desservie dans les communes rurales et les bourgs.

**Les grandes surfaces ne le sont pas.** Pour les supermarchés et les hypermarchés, les courbes sont
plates ou descendantes dans la plupart des territoires urbains : être proche d'un supermarché va même
avec des prix plus bas dans les centres urbains intermédiaires (−30 %). Les grandes surfaces sont
souvent situées dans des zones commerciales périphériques, moins recherchées pour y habiter.
Ce n'est que dans le rural que la proximité d'un supermarché est associée à des prix plus élevés.

**Deux cas particuliers.**
- Dans les grands centres urbains (en bleu), le prix double dans la dernière tranche, quel que soit
  le type de commerce : voir le graphique hors Paris ci-dessous.
- Dans les centres urbains intermédiaires (en orange), le prix *baisse* quand l'accès augmente, pour
  toutes les catégories. Les logements les mieux desservis y sont ceux des centres-villes, qui sont souvent
  moins chers que les quartiers résidentiels de périphérie dans les villes moyennes.

Ces résultats sont des associations, pas des effets causaux. Un bon accès aux commerces de proximité
va avec la centralité d'un quartier, qui fait elle-même monter les prix. Les régressions de la partie 7
contrôlent une partie de ces facteurs.

In [ ]:
graphique_tranches("prix_par_tranche_hors_paris.csv", "Même calcul sans Paris")

Sans les ventes parisiennes, le saut de la dernière tranche dans les grands centres urbains (en bleu)
est divisé par trois environ : le prix y est 40 % plus élevé que dans la tranche la moins bien desservie,
contre plus du double avec Paris (+60 % pour les supermarchés). L'essentiel de l'écart venait donc de Paris, dont
les prix s'expliquent par bien d'autres facteurs que les commerces. Les « hypermarchés »
parisiens sont d'ailleurs pour la plupart des grands magasins (partie 3).

Un écart subsiste toutefois : dans les autres grandes villes, les logements les mieux desservis
restent plus chers. Ce sont surtout ceux des centres-villes, où les commerces se concentrent et où les
prix sont plus élevés. L'accès aux commerces et la centralité du quartier vont ensemble, et ce graphique
ne permet pas de les séparer.

Les autres niveaux de densité ne changent pas, puisque Paris n'appartient qu'au premier. Les conclusions
précédentes tiennent donc : l'accès aux commerces de proximité et spécialisés est associé à des prix
plus élevés partout, alors que l'accès aux grandes surfaces ne l'est que dans le rural.

**Comparaison des deux mesures d'accessibilité**

En complément de la mesure d'accessibilité définie précédemment avec une frontière dégressive, on construit une seconde mesure avec une frontière nette.

Les deux méthodes utilisent les mêmes rayons d'accessibilité, mais diffèrent dans la pondération des commerces :

Frontière nette : chaque commerce situé dans le rayon reçoit un poids de 1, quelle que soit sa distance au logement.

Frontière dégressive : le poids de chaque commerce diminue linéairement avec sa distance au logement.

Pour comparer ces deux méthodes, nous prenons l'exemple des supermarchés. Les indices médians sont normalisés en base 100 pour les grands centres urbains (DENS7 = 1), afin de comparer leur évolution selon la densité malgré leurs échelles différentes (l'indice d'accessibilité avec une frontière nette est toujours supérieur à celui avec une frontière dégressive).

In [ ]:
montrer("accessibilite_supermarches_par_densite.png")

Avec les deux méthodes, l'accessibilité médiane aux supermarchés diminue lorsque la densité du territoire baisse.

Cependant, cette baisse est moins marquée avec la frontière dégressive pour les niveaux de densité intermédiaires (DENS7 de 2 à 5).

Cela pourrait s'expliquer par les rayons d'accessibilité plus grands dans les territoires moins denses : un supermarché peut être plus éloigné en mètres tout en conservant un poids relativement élevé dans l'indice dégressif.

Le choix de la méthode influence donc la mesure des écarts d'accessibilité entre les territoires. Les deux indicateurs seront utilisés dans les régressions suivantes afin de comparer leurs résultats.

## 7. Régressions

*Partie correspondant au notebook `04_regressions`.*

On estime six régressions linéaires pour expliquer le logarithme du prix au m².

Tous les modèles contrôlent la surface réelle bâtie, le type de logement et le département. À partir de M2, on contrôle également la densité urbaine (`DENS7`).

**Notations :**

- $P_i$ : prix au m² du logement $i$.
- $D_i$ : distance au commerce alimentaire le plus proche, toutes catégories confondues.
- $D_{ik}$ : distance au commerce le plus proche de la catégorie $k$.
- $N_{ik}$ : nombre de commerces accessibles de la catégorie $k$ (frontière nette).
- $A_{ik}$ : indice d'accessibilité de la catégorie $k$ (frontière dégressive).
- $S_i$ : surface réelle bâtie du logement.
- $\alpha_{T_i}$ : effet du type de logement.
- $\delta_{DEP_i}$ : effet du département.
- $\theta_{DENS_i}$ : effet du niveau de densité urbaine.
- $\lambda_{DENS_i}$ : coefficient d'interaction entre distance et densité.
- $\varepsilon_i$ : terme d'erreur.
- $\beta_0$ : constante ; $\beta_k$ et $\gamma$ : coefficients estimés.

Les quatre catégories $k$ sont les hypermarchés, les supermarchés, les commerces de proximité et les commerces spécialisés.

### M1 — Distance au commerce le plus proche

$$
\log(P_i) = \beta_0 + \beta_1 D_i + \gamma S_i + \alpha_{T_i} + \delta_{DEP_i} + \varepsilon_i
$$

On cherche à savoir si la distance au commerce alimentaire le plus proche est associée au prix au m², en contrôlant les caractéristiques du logement et le département.

### M2 — Distance au commerce le plus proche et densité

$$
\log(P_i) = \beta_0 + \beta_1 D_i + \gamma S_i + \alpha_{T_i} + \delta_{DEP_i} + \theta_{DENS_i} + \varepsilon_i
$$

On ajoute la densité urbaine pour voir si sa prise en compte modifie le lien entre la distance aux commerces et le prix au m².

### M3 — Interaction entre distance et densité urbaine

$$
\log(P_i) = \beta_0 + \beta_1 D_i + \theta_{DENS_i} + \lambda_{DENS_i}D_i + \gamma S_i + \alpha_{T_i} + \delta_{DEP_i} + \varepsilon_i
$$

On introduit une interaction entre la distance et la densité urbaine pour déterminer si l'association entre distance et prix immobiliers varie selon le type de territoire.

### M4 — Distance au commerce le plus proche par catégorie

$$
\log(P_i) = \beta_0 + \sum_{k=1}^{4}\beta_k D_{ik} + \gamma S_i + \alpha_{T_i} + \delta_{DEP_i} + \theta_{DENS_i} + \varepsilon_i
$$

On distingue les quatre catégories de commerces pour déterminer si la relation entre distance et prix immobiliers diffère selon le type de commerce.

### M5 — Nombre de commerces accessibles (frontière nette)

$$
\log(P_i) = \beta_0 + \sum_{k=1}^{4}\beta_k N_{ik} + \gamma S_i + \alpha_{T_i} + \delta_{DEP_i} + \theta_{DENS_i} + \varepsilon_i
$$

On étudie si le nombre de commerces accessibles dans les rayons définis est associé au prix au m², sans pondération par la distance.

### M6 — Indice d'accessibilité (frontière dégressive)

$$
\log(P_i) = \beta_0 + \sum_{k=1}^{4}\beta_k A_{ik} + \gamma S_i + \alpha_{T_i} + \delta_{DEP_i} + \theta_{DENS_i} + \varepsilon_i
$$

On cherche à savoir si l'indice d'accessibilité qui met plus de poids sur les commerces proches est associé aux prix immobiliers, et si les résultats diffèrent de ceux de la frontière nette.

Pour l'ensemble des modèles, les écarts-types des coefficients sont robustes à l'hétéroscédasticité (HC3).

Les résultats des six modèles sont importés depuis des CSV dans le dossier outputs/tables/

In [ ]:
pd.read_csv(config.DOSSIER_TABLES / "comparaison_modeles.csv")

In [ ]:
for nom in ["M1", "M2", "M3", "M4", "M5", "M6"]:
    print(nom)
    display(pd.read_csv(
        config.DOSSIER_TABLES / f"coefficients_{nom}.csv",
        index_col=0
    ))

### Principaux résultats

**M1 — Distance au commerce le plus proche (R² = 0,493)**

La distance est négativement associée au prix immobilier : **100 mètres supplémentaires correspondent à environ −0,68 % du prix au m²** (p < 0,001).

**M2 — Ajout de la densité urbaine (R² = 0,509)**

Après contrôle de la densité, cette association diminue à **−0,19 % pour 100 mètres**. Une partie de la relation observée dans M1 s'explique donc par les différences entre territoires urbains et ruraux.

**M3 — Interaction entre distance et densité (R² = 0,510)**

L'association varie selon le territoire : elle est positive dans les grands centres urbains (environ **+1,2 % pour 100 mètres**) et négative dans les espaces ruraux les moins denses.

Une explication possible est qu'en ville, un léger éloignement des commerces peut correspondre à des quartiers résidentiels plus calmes, tandis qu'en zone rurale, la proximité des commerces représente un avantage.

**M4 — Distances par catégorie (R² = 0,512)**

Les distances aux quatre catégories sont négativement associées aux prix (p < 0,001). L'association est particulièrement marquée pour les **commerces de proximité (−1,7 % par km)** et les **supermarchés (−1,3 % par km)**.

**M5 — Frontière nette (R² = 0,519)**

Un nombre plus élevé d'hypermarchés, de supermarchés et de commerces spécialisés accessibles est associé à des prix plus élevés. En revanche, le nombre de commerces de proximité présente une association négative (**−0,19 % par commerce supplémentaire**).

Cela montre que la proximité d'un commerce et la concentration de nombreux commerces ne mesurent pas la même chose.

Une forte concentration de petits commerces pourrait s'accompagner de davantage de nuisances, tandis que la présence d'hypermarchés pourrait être associée à des territoires mieux équipés.

**M6 — Frontière dégressive (R² = 0,517)**

Les coefficients conservent les mêmes signes que dans M5. La pondération par la distance ne modifie donc pas les principaux résultats. La frontière nette présente un ajustement légèrement meilleur (**0,519 contre 0,517**).

Finalement , les six modèles montrent que **l'accessibilité aux commerces alimentaires est statistiquement associée aux prix immobiliers**, mais que cette relation dépend de la densité urbaine et du type de commerce.

La proximité commerciale semble davantage valorisée dans les territoires ruraux, tandis que dans les grands centres urbains, un léger éloignement peut être associé à des prix plus élevés.

## 8. Limites et pistes d'amélioration

**Des associations, pas des effets causaux.** Un bon accès aux commerces va avec la centralité d'un
quartier, son ancienneté et ses services, qui font eux aussi monter les prix. Comparer les logements à densité
égale et contrôler par le département réduit ce problème sans l'éliminer.

**Des distances à vol d'oiseau.** La distance est mesurée en ligne droite, pas par la route. Elle
sous-estime l'éloignement réel, surtout en montagne ou de part et d'autre d'un fleuve ou d'une autoroute.

**Une nomenclature imparfaite.** La catégorie « hypermarché » de la BPE inclut les grands magasins,
et la base compte des établissements plutôt que des magasins. On l'a constaté à Paris et traité en
présentant les résultats hors Paris.

**Des corrélations entre catégories de commerces.** Les commerces ont tendance à se regrouper (clusters) : un hypermarché peut, par exemple, être entouré de nombreux commerces spécialisés. Cette multicolinéarité peut compliquer l'interprétation des coefficients de M4, M5 et M6.

**Des informations manquantes sur les logements.** La DVF ne donne ni l'état du logement, ni l'étage,
ni la performance énergétique, qui pèsent beaucoup sur le prix.

**Un champ limité.** Une seule année (2025), en France métropolitaine hors Alsace-Moselle, que la DVF
ne couvre pas.

**Pistes.** 

Mesurer des temps de trajet sur le réseau routier, ajouter d'autres équipements (écoles,
gares, services de santé) pour mieux isoler l'effet des commerces, utiliser les revenus des habitants
par quartier (données Filosofi de l'INSEE) pour tenir compte du niveau de vie, et étendre l'analyse
à plusieurs années.

Distinguer deux zones de proximité : dans les grands centres urbains, compter séparément les commerces situés à moins de $r$ mètres et ceux situés entre $r$ et $R$ mètres ($R > r$). L'hypothèse serait que les commerces très proches sont associés à des prix plus faibles en raison des nuisances, tandis que ceux un peu plus éloignés sont davantage valorisés, car ils restent accessibles sans être directement gênants.

## 9. Reproduire les résultats

Cette synthèse ne fait que lire les résultats enregistrés dans `outputs/`. Pour tout recalculer depuis
les données publiques, il faut lancer les notebooks dans l'ordre :

1. `notebooks/01_nettoyage_bpe.ipynb` : commerces alimentaires, cartes
2. `notebooks/02_nettoyage_dvf.ipynb` : ventes de logements, niveaux de densité
3. `notebooks/03_croisement.ipynb` : distances, indice d'accessibilité, tableaux descriptifs
4. `notebooks/04_regressions.ipynb` : modèles

Les données sont téléchargées automatiquement depuis leurs sources (INSEE et data.gouv.fr). Le détail
de l'installation est dans le `README.md`.

## 10. Ce que nous avons appris

**Sur les données.** Une base publique n'est jamais « prête à l'emploi ». Plusieurs de nos problèmes
venaient d'une mauvaise compréhension de la structure des données plutôt que d'erreurs de code :
une vente DVF répartie sur plusieurs lignes, un prix total répété sur chaque logement, des grands
magasins rangés parmi les hypermarchés, des établissements comptés à la place des magasins. Nous avons
appris à confronter systématiquement les résultats à ce que nous connaissions du terrain : un prix
médian absurde à Crégy-lès-Meaux ou un seul commerce dans le 75116 étaient des signaux d'alerte, pas
des curiosités.

**Sur la méthode.** Les choix de nettoyage ne sont pas neutres. Des seuils fixes (1 €, 50 millions
d'euros) semblaient naturels, mais n'avaient pas de sens à l'échelle d'un pays où les prix varient d'un
facteur 100. L'ordre des filtres changeait les résultats (Vitreux). Et un filtre qui paraissait
raisonnable retirait en fait de vraies ventes (Lourdes). Nous avons appris à justifier chaque seuil, et à
vérifier ce qu'un filtre retire avant de l'adopter. De même, la construction de l'indice d'accessibilité
est passée par plusieurs versions avant d'aboutir à une mesure comparable d'un territoire à l'autre.

**Sur les outils.** Nous avons découvert le travail sur des données géographiques avec `geopandas` :
choisir une projection adaptée au calcul de distances (Lambert 93), faire des jointures spatiales, et
utiliser des structures adaptées (arbre de recherche `cKDTree`) pour calculer des distances sur des
centaines de milliers de points en quelques secondes.

**Sur la méthode de travail : regarder avant d'agir.** Notre principale erreur a été de nous lancer
dans le nettoyage avant d'avoir vraiment regardé les données. Nous avons supprimé des colonnes, des
valeurs manquantes et des ventes « suspectes » avant de comprendre la structure de la DVF. Nous avons
d'abord travaillé sur une version sans coordonnées, puis fixé des seuils de prix arbitraires, et testé un
filtre sur les ventes groupées qui retirait de vraies ventes. Une grande partie de ce travail a ensuite
dû être supprimée ou corrigée. L'indice d'accessibilité a suivi le même chemin : nous avons commencé à
le coder avant d'avoir défini précisément ce qu'il devait mesurer. Nous sommes ainsi passés par un
barycentre des commerces, une moyenne pondérée par l'inverse de la distance et un score par commerce,
avant d'aboutir au rayon par niveau de densité, à la décroissance linéaire et au rang. Avec le recul,
il aurait fallu commencer par explorer chaque base (aperçu des lignes, valeurs prises par chaque
variable, cartes), puis écrire noir sur blanc ce que l'on voulait mesurer et pourquoi, et seulement
ensuite coder. Le temps passé à regarder les données est largement regagné ensuite.

**Sur le travail à deux.** Nous avons utilisé Git avec des branches et des *pull requests*, mais sans
toujours en respecter la logique. Une branche n'a d'intérêt que si chacun y travaille sur des fichiers
distincts. Sur notre branche de correction, nous avons modifié tous les deux les mêmes notebooks en
parallèle, ce qui a produit des conflits difficiles à résoudre : un notebook est un fichier JSON que Git
fusionne très mal, et il a fallu comparer les versions cellule par cellule pour reconstituer la bonne.
Nous avons aussi perdu une partie du notebook de synthèse, faute de l'avoir enregistrée et commitée.
Les règles que nous en retenons sont simples : répartir les fichiers à l'avance (un notebook, un
responsable), récupérer la version commune (`pull`) avant de commencer, et faire des commits petits et
fréquents. Les branches sont réservées aux modifications importantes, chacun sur la sienne.